**Amazon S3 (Simple Storage Service)** is AWS's object storage service — infinitely scalable, highly durable, and foundational to virtually every AWS architecture. It stores files (called objects) in containers (called buckets) and serves them over HTTP.

S3 is one of the most heavily tested services on the Solutions Architect exam — storage classes, security, versioning, performance, and access patterns all appear regularly.

## Buckets and Objects

### Buckets
A **bucket** is a container for objects. Key rules:
- Bucket names are **globally unique** across all AWS accounts and regions
- Names must be 3–63 characters, lowercase, no underscores, no IP-format names
- Buckets are created in a specific region — data never leaves that region unless you explicitly replicate it
- No limit on the number of objects in a bucket

### Objects
An **object** is a file plus its metadata. Key facts:

| Property | Detail |
|---|---|
| **Key** | The full "path" — e.g. `images/2024/photo.jpg`. S3 is flat; the `/` is just part of the key name |
| **Value** | The actual bytes of the file |
| **Max size** | 5 TB per object |
| **Min size** | 0 bytes |
| **Upload limit** | Single PUT max 5 GB; use multipart upload for > 100 MB |
| **Metadata** | System metadata (content-type, ETag) + up to 2 KB user-defined metadata |
| **Version ID** | Present if versioning is enabled on the bucket |

### S3 is not a filesystem
S3 has no real directories. `images/photo.jpg` and `images/thumb.jpg` are two separate objects with keys that happen to share a prefix. The `/` is cosmetic — S3 displays it as folders in the console.

## Storage Classes

S3 offers multiple storage classes with different trade-offs between cost, availability, and retrieval time.

| Storage Class | Availability | Min Duration | Retrieval | Use Case |
|---|---|---|---|---|
| **S3 Standard** | 99.99% | None | Immediate | Frequently accessed data |
| **S3 Intelligent-Tiering** | 99.9% | None | Immediate (frequent/infrequent); minutes (archive) | Unknown or changing access patterns |
| **S3 Standard-IA** | 99.9% | 30 days | Immediate | Infrequent access, rapid retrieval |
| **S3 One Zone-IA** | 99.5% | 30 days | Immediate | Infrequent access, non-critical, single AZ |
| **S3 Glacier Instant Retrieval** | 99.9% | 90 days | Immediate (ms) | Archives accessed once per quarter |
| **S3 Glacier Flexible Retrieval** | 99.99% | 90 days | 1–12 hrs (standard); free bulk | Backups, disaster recovery |
| **S3 Glacier Deep Archive** | 99.99% | 180 days | 12–48 hrs | Long-term compliance archives |

### Key exam distinctions

- **Standard-IA vs One Zone-IA**: One Zone-IA is 20% cheaper but stored in a single AZ — data is lost if the AZ is destroyed. Use only for reproducible data (thumbnails, derived data).
- **Glacier Instant vs Flexible**: Instant gives millisecond retrieval like Standard; Flexible is cheaper but takes minutes to hours.
- **Intelligent-Tiering**: Automatically moves objects between tiers based on access patterns. No retrieval fee, but a small monthly monitoring fee per object. Best when access patterns are unpredictable.
- **Minimum duration charges**: Even if you delete an object before the minimum duration, you are charged for the full minimum period.

## S3 Security

S3 has multiple layers of access control. By default, all buckets and objects are **private**.

### 1. Bucket Policies
JSON-based resource policies attached directly to a bucket. They can grant or deny access to:
- Specific IAM users, roles, or accounts
- Everyone (public access — requires Block Public Access to be disabled)
- Specific services (e.g. CloudFront OAC)

```json
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Principal": "*",
    "Action": "s3:GetObject",
    "Resource": "arn:aws:s3:::my-bucket/*",
    "Condition": {
      "StringEquals": { "aws:SourceVpce": "vpce-1234abcd" }
    }
  }]
}
```

### 2. IAM Policies
Identity-based policies on users/roles that grant S3 permissions. Works for same-account access.

### 3. Block Public Access
A four-setting safety override that blocks public access regardless of bucket policy or ACLs. Can be set at the account level (applies to all buckets) or per bucket. **Enabled by default for new buckets.**

### 4. ACLs (legacy)
Object-level or bucket-level access control lists. Mostly disabled and superseded by bucket policies. Leave disabled unless you have a specific need.

### Access decision flow
```
Request arrives
  │
  ├── Block Public Access enabled? → DENY (even if policy allows)
  ├── Explicit Deny in any policy? → DENY
  ├── IAM policy allows? + Bucket policy allows? → ALLOW (cross-account)
  └── Either IAM or bucket policy allows (same account)? → ALLOW
```

## S3 Encryption

### Server-Side Encryption (SSE)

| Type | Key managed by | Key stored | Notes |
|---|---|---|---|
| **SSE-S3** | AWS | AWS (opaque) | Default since Jan 2023; no extra cost; AES-256 |
| **SSE-KMS** | AWS KMS | KMS | Audit trail in CloudTrail; customer controls key rotation; KMS API call cost |
| **SSE-C** | You | You (send per request) | You manage the key; AWS never stores it; must use HTTPS |

### Client-Side Encryption
You encrypt data before uploading. S3 stores the ciphertext. You manage all key operations. Use when you need end-to-end encryption without trusting AWS at all.

### Encryption in transit
S3 supports both HTTP and HTTPS. Always enforce HTTPS using a bucket policy condition:
```json
{ "Effect": "Deny", "Action": "s3:*", "Resource": "...",
  "Condition": { "Bool": { "aws:SecureTransport": "false" } } }
```

> **SSE-S3 is the default** since Jan 2023 — all new objects are encrypted automatically with no configuration needed.

## Versioning

With versioning enabled, S3 preserves every version of an object. Overwriting or deleting an object does not destroy previous versions.

```
PUT photo.jpg  → version ID: abc123
PUT photo.jpg  → version ID: def456  (latest)
DELETE photo.jpg → adds a delete marker  (latest = delete marker)
  └── Previous versions still exist and are retrievable
```

### Key behaviours
- Once enabled, versioning **cannot be disabled** — only suspended
- Objects uploaded before versioning was enabled get a `null` version ID
- Deleting an object without specifying a version ID adds a **delete marker** — it does not permanently delete
- Permanently delete by specifying the exact version ID
- Storage costs accumulate for all versions — use lifecycle rules to expire old versions

### MFA Delete
Requires MFA authentication to permanently delete object versions or change the versioning state. Provides an extra layer of protection against accidental or malicious deletion. Can only be enabled by the root account.

## S3 Performance

### Baseline throughput
S3 automatically scales to high request rates:
- **3,500 PUT/COPY/POST/DELETE** requests per second per prefix
- **5,500 GET/HEAD** requests per second per prefix

Spread objects across multiple prefixes to multiply throughput (e.g. `a/`, `b/`, `c/` prefixes give 3× GET throughput).

### Multipart Upload
Splits large objects into parts, uploads them in parallel, then S3 assembles them.
- **Recommended** for objects > 100 MB
- **Required** for objects > 5 GB
- Each part: 5 MB–5 GB; last part can be smaller
- If a part fails, retry only that part — not the whole upload

### S3 Transfer Acceleration
Routes uploads through the nearest **CloudFront edge location** and then over AWS's private backbone to the S3 bucket. Speeds up uploads from distant locations.

```
User (Tokyo) → CloudFront Edge (Tokyo) → AWS backbone → S3 (us-east-1)
vs.
User (Tokyo) → public internet → S3 (us-east-1)   ← slower
```

### Byte-Range Fetches
Request specific byte ranges of an object in parallel — effectively multipart download. Speeds up large downloads and allows partial retrieval (e.g. read just the header of a large file).

## Access Patterns

### Pre-signed URLs
A time-limited URL that grants temporary access to a specific object — without requiring the recipient to have AWS credentials.

- Generated by an IAM identity (user or role) with access to the object
- The URL inherits the permissions of the generating identity
- Default expiry: 1 hour; max: 7 days (via STS credentials)
- Use cases: share a private file, allow a user to upload directly to S3 without going through your backend

### S3 Access Points
Named network endpoints attached to a bucket, each with its own access policy. Simplify access management when many applications share one bucket.

```
my-data-bucket
  ├── Access Point: analytics-ap  → allows data-science team read access to /data/*
  ├── Access Point: app-ap        → allows app role read/write to /uploads/*
  └── Access Point: vpc-ap        → VPC-only access, no internet
```

### S3 Object Lambda
Transform object data on the fly as it is retrieved — without storing a modified copy. Use cases: redact PII, resize images, convert formats, enrich data.

```
App → S3 Object Lambda Access Point → Lambda (transforms data) → original S3 object
```

## Static Website Hosting

S3 can serve a static website (HTML, CSS, JS, images) directly — no servers needed.

Setup:
1. Enable static website hosting on the bucket
2. Set index document (`index.html`) and error document (`error.html`)
3. Disable Block Public Access and add a bucket policy granting `s3:GetObject` to `*`
4. Access via the S3 website endpoint: `bucket-name.s3-website.region.amazonaws.com`

### S3 + CloudFront for production
For a production static site, put CloudFront in front of S3:
- Custom domain + HTTPS (ACM certificate)
- Global CDN caching for low latency
- **Origin Access Control (OAC)** — keep the bucket private, only CloudFront can read it
- WAF protection at the edge

> With OAC, the bucket does not need to be public — CloudFront authenticates to S3 using its own IAM-like identity.

## Working with S3 via boto3

In [ ]:
import boto3
import json

s3 = boto3.client('s3', region_name='us-east-1')
s3_resource = boto3.resource('s3')

In [ ]:
# Create a bucket with versioning and SSE-KMS encryption
bucket_name = 'my-demo-bucket-unique-123'

s3.create_bucket(Bucket=bucket_name)

# Enable versioning
s3.put_bucket_versioning(
    Bucket=bucket_name,
    VersioningConfiguration={'Status': 'Enabled'}
)

# Enforce SSE-KMS encryption
s3.put_bucket_encryption(
    Bucket=bucket_name,
    ServerSideEncryptionConfiguration={
        'Rules': [{
            'ApplyServerSideEncryptionByDefault': {
                'SSEAlgorithm': 'aws:kms'
            },
            'BucketKeyEnabled': True   # reduces KMS API call cost
        }]
    }
)
print(f"Bucket {bucket_name} created with versioning and SSE-KMS")

In [ ]:
# Upload objects with different storage classes
s3.put_object(
    Bucket=bucket_name,
    Key='frequent/data.json',
    Body=b'{"status": "active"}',
    StorageClass='STANDARD'
)

s3.put_object(
    Bucket=bucket_name,
    Key='archive/old-data.json',
    Body=b'{"status": "archived"}',
    StorageClass='GLACIER_IR'   # Glacier Instant Retrieval
)
print("Objects uploaded")

In [ ]:
# Generate a pre-signed URL valid for 1 hour
url = s3.generate_presigned_url(
    ClientMethod='get_object',
    Params={'Bucket': bucket_name, 'Key': 'frequent/data.json'},
    ExpiresIn=3600
)
print(f"Pre-signed URL (valid 1h):\n{url}")

In [ ]:
# Generate a pre-signed POST URL for direct browser upload
response = s3.generate_presigned_post(
    Bucket=bucket_name,
    Key='uploads/${filename}',
    Fields={'Content-Type': 'image/jpeg'},
    Conditions=[
        ['content-length-range', 1, 10485760]  # 1B to 10MB
    ],
    ExpiresIn=3600
)
print("POST URL:", response['url'])
print("Fields:", response['fields'])

In [ ]:
import os

# Multipart upload for large files
def multipart_upload(file_path, bucket, key, part_size_mb=10):
    part_size = part_size_mb * 1024 * 1024
    mpu = s3.create_multipart_upload(Bucket=bucket, Key=key)
    upload_id = mpu['UploadId']
    parts = []

    with open(file_path, 'rb') as f:
        part_number = 1
        while True:
            data = f.read(part_size)
            if not data:
                break
            response = s3.upload_part(
                Bucket=bucket, Key=key,
                PartNumber=part_number, UploadId=upload_id, Body=data
            )
            parts.append({'PartNumber': part_number, 'ETag': response['ETag']})
            print(f"Uploaded part {part_number}")
            part_number += 1

    s3.complete_multipart_upload(
        Bucket=bucket, Key=key, UploadId=upload_id,
        MultipartUpload={'Parts': parts}
    )
    print(f"Upload complete: s3://{bucket}/{key}")

# multipart_upload('large-file.bin', bucket_name, 'uploads/large-file.bin')

In [ ]:
# List all versions of objects in the bucket
response = s3.list_object_versions(Bucket=bucket_name)

print("Versions:")
for v in response.get('Versions', []):
    print(f"  {v['Key']:<40} {v['VersionId'][:8]}  {'LATEST' if v['IsLatest'] else ''}")

print("\nDelete markers:")
for dm in response.get('DeleteMarkers', []):
    print(f"  {dm['Key']:<40} {dm['VersionId'][:8]}  delete marker")

In [ ]:
# Enforce HTTPS-only access via bucket policy
policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Deny",
        "Principal": "*",
        "Action": "s3:*",
        "Resource": [
            f"arn:aws:s3:::{bucket_name}",
            f"arn:aws:s3:::{bucket_name}/*"
        ],
        "Condition": {"Bool": {"aws:SecureTransport": "false"}}
    }]
}
s3.put_bucket_policy(Bucket=bucket_name, Policy=json.dumps(policy))
print("HTTPS-only policy applied")

## Summary

| Concept | Key Takeaway |
|---|---|
| Bucket | Globally unique name; region-specific; unlimited objects |
| Object | Key + value; max 5 TB; use multipart for > 100 MB |
| S3 Standard | Default; frequent access; highest availability |
| Standard-IA | Infrequent access; retrieval fee; 30-day minimum |
| One Zone-IA | Single AZ; 20% cheaper; only for reproducible data |
| Glacier Instant | Millisecond retrieval; 90-day minimum; quarterly access |
| Glacier Flexible | Hours retrieval; cheaper; backups and DR |
| Glacier Deep Archive | 12–48hr retrieval; cheapest storage; compliance archives |
| Intelligent-Tiering | Auto-moves objects; no retrieval fee; monitoring fee per object |
| Block Public Access | Account or bucket level; overrides all policies; on by default |
| SSE-S3 | Default encryption; AWS manages keys; no extra cost |
| SSE-KMS | Audit trail; customer controls key; KMS API cost |
| Versioning | Preserves all versions; delete adds marker; cannot be disabled |
| Multipart upload | Required > 5 GB; recommended > 100 MB; retry parts independently |
| Transfer Acceleration | Routes through CloudFront edge; speeds up distant uploads |
| Pre-signed URL | Temporary access to private objects; max 7 days |
| Static website + CloudFront | Use OAC to keep bucket private; custom domain + HTTPS via ACM |